# 0-shot to get outcome probs

## Setr Up

In [ ]:
import os
import sys

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH
from src.utils import IMPACTME_OUTCOMES_FULL, OUTCOMES_EXPAND, IMPACTME_OUTCOMES_SHOT
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 500)
print(f"BASE PATH: {BASE_PATH}")

Load model

In [ ]:
## MODEL
model_dir = Path("/lscratch3/models_llama/")
models = []
for idx, entry in enumerate(model_dir.iterdir()):
    print(f"{idx} \t {entry.name}")
    models.append(entry.name)

In [ ]:
CUR_MODEL = models[6]
MODEL_ID = model_dir / CUR_MODEL
print(f"Using model: {CUR_MODEL}")
assert Path(MODEL_ID).exists()

## Run Impact Types (self, parent) in parallel

In [ ]:
CONFIG = {
    "model_id": str(MODEL_ID),
    "temp": 0.6,
    # "temp": 1.0, #~ or worse than 0.6 in practice
    "top_p": 0.6,
}
IMPACT_TYPES = ["self", "parent"]

Run

In [ ]:
import subprocess
import json
import signal


def make_cmd(
    cur_model,
    outcome_list,
    impact_type_list,
    base_df_dir,
    json_config,
    gpu_group,
    outcome_desc_type,
    n_shot,
    output_type,
):
    temp = json_config["temp"]
    log_dir = (
        BASE_PATH
        / "logs"
        / f"shot_{n_shot}"
        / f"temp_{temp}"
        / output_type
        / cur_model
        / outcome_desc_type
    )
    save_dir = (
        BASE_PATH
        / "data"
        / f"w_{n_shot}_shot_probs"
        / f"temp_{temp}"
        / output_type
        / cur_model
        / outcome_desc_type
    )
    cmd = [
        "python",
        "src/shot_controller.py",
        "--gpu-groups",
        *gpu_group,
        "--outcomes",
        *outcome_list,
        "--impact_types",
        *impact_type_list,
        "--llm_config",
        json.dumps(json_config),
        "--base_df_dir",
        str(base_df_dir),
        "--save_dir",
        str(save_dir),
        "--log_dir",
        str(log_dir),
        "--outcome_desc_type",
        outcome_desc_type,
        "--n_shot",
        str(n_shot),
        "--output_type",
        output_type,
    ]
    return cmd

In [ ]:
def get_outcome_list(desc_type):
    if desc_type == "original":
        return IMPACTME_OUTCOMES_SHOT
    elif desc_type == "expanded":
        return OUTCOMES_EXPAND
    elif desc_type == "combined":
        return OUTCOMES_EXPAND + IMPACTME_OUTCOMES_SHOT
    else:
        raise ValueError(f"Unrecognized desc_type: {desc_type}")

In [ ]:
### Set env
env = os.environ.copy()
env["PYTHONPATH"] = str(BASE_PATH)
if "70B" in CUR_MODEL.upper():
    gpu_groups = [
        "0,1,2,3",
        "4,5,6,7",
    ]
else:
    gpu_groups = [
        "0,1",
        "2,3",
        "4,5",
        "6,7",
    ]

base_df_dir = BASE_PATH / "data" / "cleaned" / "question_answer"
outcome_desc_type = (
    "expanded"  # one of neutral, original, augment, augment_neg, new, expanded
)
outcome_list = get_outcome_list(outcome_desc_type)
cmd = make_cmd(
    cur_model=CUR_MODEL,
    outcome_list=outcome_list,
    impact_type_list=IMPACT_TYPES,
    base_df_dir=base_df_dir,
    json_config=CONFIG,
    gpu_group=gpu_groups,
    outcome_desc_type=outcome_desc_type,
    n_shot=0,  # 3 max
    output_type="proba-limited",  # one of binary, proba, proba-limited
)
proc = subprocess.Popen(
    cmd,
    cwd=BASE_PATH,
    start_new_session=True,
    env=env,
)
print("Controller PID:", proc.pid)

In [ ]:
proc.poll()

In [ ]:
os.killpg(proc.pid, signal.SIGTERM)  # gracefull shtudown

In [ ]:
os.killpg(proc.pid, signal.SIGKILL)  # force kill

# Check output

## Look at generations

In [ ]:
import torch
import transformers

transformers.logging.set_verbosity_error()
tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_ID)
model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map="balanced",
)

In [ ]:
# shot.py
import os, sys

sys.path.append(os.path.abspath("../"))
from src.config import BASE_PATH

import json
import argparse
import math
import logging
from pathlib import Path
import pandas as pd

import torch.nn.functional as F
import numpy as np
from src.shot import get_results_proba


def get_current_prob(
    shot_type, llama_name, desc_type, impact_type, id_str, outcome, tokenizer, model
):
    import_path = (
        BASE_PATH
        / "data"
        / f"w_{0}_shot_probs_old"
        / f"temp_{0.6}"
        / shot_type
        / llama_name
        / desc_type
        / f"{impact_type}.parquet"
    )
    if shot_type == "binary":
        shot_col = "open_text_block_0_shot_binary"
    else:
        if desc_type == "original":
            shot_col = "open_text_block_0_shot"
        else:
            shot_col = "open_text_block_0_shot_proba_limited"
    # shot_col = "open_text_block_0_shot_proba_limited"
    dummy = pd.read_parquet(import_path)
    dummy = dummy[dummy["open_text_block"].str.contains(id_str)]
    for row in dummy.itertuples():
        impact_score = row.impact_score_parent
        open_text_block = row.open_text_block
        print(impact_score)
        print(open_text_block)
        print(getattr(row, shot_col))
    ## Generate prob from model
    output = generate_prob(desc_type, impact_type, id_str, outcome, tokenizer, model)
    return output


def generate_prob(desc_type, impact_type, dawba_id, outcome, tokenizer, model):
    import_path = (
        BASE_PATH / "data" / "cleaned" / "question_answer" / f"{impact_type}.parquet"
    )
    text_df = pd.read_parquet(import_path)
    text_df_sub = text_df[text_df["DAWBA_ID"] == dawba_id]
    # text_df_sub=text_df.head(1)
    # text_df_sub["open_text_block"] = (
    #     "Question: What is your experience with your child? Answer: He doesn't like to go to schoool."
    # )
    output = get_results_proba(
        tokenizer=tokenizer,
        model=model,
        text_df=text_df_sub,
        outcome_names=[outcome],
        outcome_desc_type=desc_type,
        n_shot=0,
        temp=0.6,
        impact_type=impact_type,
        limit_output=True,
    )
    print(output["open_text_block_0_shot"])

In [ ]:
shot_type = "proba-limited"
llama_name = CUR_MODEL
desc_type = "new"
impact_type = "parent"
dawba_id = 234891
outcome = "f2"
# output = get_current_prob(
#     shot_type, llama_name, desc_type, impact_type, id_str, "f2", tokenizer, model
# )
generate_prob(desc_type, impact_type, dawba_id, outcome, tokenizer, model)

In [ ]:
r = pd.read_parquet(
    "/home/mataray/DAWBA/data/w_0_shot_probs/temp_0.6/proba-limited/Llama-3.3-8B-Instruct/neutral/parent.parquet"
)
r["open_text_block_0_shot_proba_limited"]

## Look at correlations

In [ ]:
from src.utils import IMPACTME_OUTCOMES_FULL, OUTCOMES_EXPAND, IMPACTME_OUTCOMES_SHOT
import seaborn as sns
import matplotlib.pyplot as plt


def extract_outcome_probs(row, outcome_list, sep_cols_flag=False):
    if sep_cols_flag:
        return {outcome: getattr(row, outcome) for outcome in outcome_list}
    prob_dict = {}
    prob_vec = row.open_text_block_0_shot
    assert len(prob_vec) == len(
        outcome_list
    ), f"""Prob vec: {len(prob_vec)}\nImpact List: {len(outcome_list)}"""
    for idx, outcome in enumerate(outcome_list):
        if sep_cols_flag:
            prob_dict[outcome] = getattr(row, outcome)
        else:
            prob_dict[outcome] = float(prob_vec[idx])
    return prob_dict


def get_outcome_list(desc):
    if desc == "expanded":
        return OUTCOMES_EXPAND
    elif desc == "new":
        return IMPACTME_OUTCOMES_FULL
    elif desc == "combined":
        return [f"{outcome}_impactme" for outcome in IMPACTME_OUTCOMES_SHOT] + [
            f"{outcome}_expand" for outcome in OUTCOMES_EXPAND
        ]
    else:
        return IMPACTME_OUTCOMES_SHOT


def get_df_w_probs(impact, llama, output_type, desc, temp, imp_dir, sep_cols_flag):
    imp_path = (
        imp_dir / f"temp_{temp}" / output_type / llama / desc / f"{impact}.parquet"
    )
    df_base = pd.read_parquet(imp_path)
    outcome_list = get_outcome_list(desc)
    new_df_rows = []
    for row in df_base.itertuples():
        prob_dict = extract_outcome_probs(
            row, outcome_list, sep_cols_flag=sep_cols_flag
        )
        new_df_rows.append(
            {
                "DAWBA_ID": row.DAWBA_ID,
                "text": row.open_text_block,
                "impact": getattr(row, f"impact_score_{impact}"),
                **prob_dict,
            }
        )
    df_w_probs = pd.DataFrame(new_df_rows)
    return df_w_probs, outcome_list


def get_hists(df_w_probs, outcome_list, temp):
    for outcome in outcome_list:
        df_w_probs[outcome].hist()
        plt.title(f"{outcome}; temp={temp}")
        plt.xlim((-0.9, 1.1))
        plt.show()
        plt.close()

In [ ]:
imp_dir = BASE_PATH / "data/w_0_shot_probs_sep_cols"
desc = "expanded"
outcome_list = get_outcome_list(desc)
df_w_probs, outcome_list = get_df_w_probs(
    impact="parent",
    llama="Llama-3.3-70B-Instruct",
    output_type="proba-limited",
    desc="expanded",
    temp=0.6,
    imp_dir=imp_dir,
    sep_cols_flag=True,
)
get_hists(df_w_probs, outcome_list, 0.6)

In [ ]:
imp_dir = BASE_PATH / "data/w_0_shot_probs_sep_cols"
desc = "expanded"
outcome_list = get_outcome_list(desc)
df_w_probs, outcome_list = get_df_w_probs(
    impact="parent",
    llama="Llama-3.3-70B-Instruct",
    output_type="proba-limited",
    desc="expanded",
    temp=1.0,
    imp_dir=imp_dir,
    sep_cols_flag=True,
)
get_hists(df_w_probs, outcome_list, temp=1.0)

Correlation matrix (outcome pairwise)

In [ ]:
corr_matrix = df_w_probs[outcome_list].corr(method="spearman")

In [ ]:
plt.figure(figsize=(40, 40))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.tight_layout()
plt.show()

Correlations (each outcome w/ impact)

In [ ]:
from scipy.stats import spearmanr

corr_rows = []
for col in outcome_list:
    r, p = spearmanr(df_w_probs[col], df_w_probs["impact"])
    corr_rows.append(
        {"outcome": col, "r": round(r, 3), "p": round(p, 4), "r_abs": abs(round(r, 3))}
    )
impact_corr_df = pd.DataFrame(corr_rows)
impact_corr_df = impact_corr_df.sort_values("r_abs", ascending=False).reset_index(
    drop=True
)
display(impact_corr_df.head(50))
plt.scatter(impact_corr_df.index, impact_corr_df["r_abs"])

In [ ]:
chosen_outcomes = impact_corr_df.iloc[:31]["outcome"].to_list()
print(chosen_outcomes)
corr_matrix = df_w_probs[chosen_outcomes].corr(method="spearman")

In [ ]:
plt.figure(figsize=(40, 40))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.tight_layout()
plt.show()

Text --> prob

In [ ]:
outcome = "e2"
df_sub = df_w_probs[df_w_probs["text"].str.contains("family")]
# df_sub = df_w_probs
df_sub = df_sub.sort_values(outcome, ascending=False)
for row in df_sub.itertuples():
    print(f"Prob: {getattr(row, outcome)}")
    print(f"Impact: {row.impact}")
    text_ = row.text.replace("family", "---- FAMILY ------")
    print(text_)

# Organize Probs

In [ ]:
TEMP = 0.6

## Combine original/expanded

In [ ]:
import numpy as np

imp_dir = BASE_PATH / f"data/w_0_shot_probs/temp_{TEMP}"
for output_dir in imp_dir.iterdir():
    for llama_dir in output_dir.iterdir():
        original_dir = llama_dir / "original"
        expanded_dir = llama_dir / "expanded"
        for impact_type in ["self", "parent"]:
            # Load dfs
            original_df = pd.read_parquet(original_dir / f"{impact_type}.parquet")
            expanded_df = pd.read_parquet(expanded_dir / f"{impact_type}.parquet")
            # Combine text prob vectors
            merged_df = original_df.copy()
            merged_df["open_text_block_0_shot"] = [
                np.concatenate([a, b])
                for a, b in zip(
                    original_df["open_text_block_0_shot"],
                    expanded_df["open_text_block_0_shot"],
                )
            ]
            # Export
            export_path = llama_dir / f"combined/{impact_type}.parquet"
            if export_path.exists():
                print(f"Over-writing file at path: {export_path}")
                export_path.unlink()
            export_path.parent.mkdir(exist_ok=True, parents=True)
            merged_df.to_parquet(export_path)
            assert export_path.exists()

## Make probs individual cols

In [ ]:
def expand_shot_output(desc, og_df):
    df = og_df.copy()
    if desc == "expanded":
        col_names = OUTCOMES_EXPAND
    elif desc == "new":
        col_names = IMPACTME_OUTCOMES_FULL
    elif desc == "combined":
        col_names = [f"{col}_impactme" for col in IMPACTME_OUTCOMES_SHOT] + [
            f"{col}_expand" for col in OUTCOMES_EXPAND
        ]
    else:
        col_names = IMPACTME_OUTCOMES_SHOT

    new_cols = pd.DataFrame(
        df["open_text_block_0_shot"].tolist(), columns=col_names, index=df.index
    )

    df = df.join(new_cols)
    df = df.drop(columns="open_text_block_0_shot")
    return df

In [ ]:
original_dir = BASE_PATH / f"data/w_0_shot_probs/temp_{TEMP}"
new_dir = BASE_PATH / f"data/w_0_shot_probs_sep_cols/temp_{TEMP}"
for output_dir in original_dir.iterdir():
    for llama_dir in output_dir.iterdir():
        for desc_dir in llama_dir.iterdir():
            for impact_path in desc_dir.iterdir():
                original_df = pd.read_parquet(impact_path)
                new_df = expand_shot_output(desc=desc_dir.name, og_df=original_df)
                new_path = (
                    new_dir
                    / output_dir.name
                    / llama_dir.name
                    / desc_dir.name
                    / impact_path.name
                )
                if new_path.exists():
                    new_path.unlink()
                new_path.parent.mkdir(exist_ok=True, parents=True)
                new_df.to_parquet(new_path)
                assert new_path.exists()